# EARLI: PDPTW Training & Evaluation
## Primary Comparison: Pure cuOpt vs. RL-Initialized cuOpt
## Secondary Comparison: PPO vs. Tree-Based Training Methods

## Setup

### Install

Run the following steps **only if EARLI is not yet installed**.

In [ ]:
# Verify Python>=3.10
import sys
print(sys.version)

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Install build tools (only if not already installed)
# !apt-get update -y && apt-get install -y git build-essential ninja-build

In [ ]:
!pip install --upgrade --extra-index-url https://pypi.nvidia.com -c constraints.txt .

### Import

In [ ]:
import os
import pickle as pkl
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
import yaml
import torch

from earli.train import train
from earli import main as earli_main
from earli.cuopt_injection_vrptw import main as cuopt_main
from earli.utils import analysis_utils as utils
from earli.utils.nv import verify_consistent_config
from earli.pdptw import PDPTW


## Prepare Datasets

**Training data**: Li&Lim 100-customer benchmark (small scale).

**Test data**: Li&Lim 1000-customer benchmark — used **exclusively for testing**, never for training or validation.

In [ ]:
import os
os.makedirs('datasets', exist_ok=True)

# Parse 100-customer Li&Lim benchmark -> training dataset
!python -m earli.benchmark_parser pdptw "li&lim benchmark/pdp_100" \
        datasets/pdptw_100.pkl

# Parse 1000-customer Li&Lim benchmark -> test-only dataset
# IMPORTANT: these instances must NOT be used for training or validation.
!python -m earli.benchmark_parser pdptw "li&lim benchmark/pdptw1000" \
        datasets/pdptw_1000.pkl


In [ ]:
!ls -lh datasets/pdptw_100.pkl datasets/pdptw_1000.pkl

## Train (Secondary Focus)

Two models are trained using the same n=100 training data:
1. **PPO** — Stable-Baselines3 on-policy rollout training.
2. **tree_based** — Tree-search guided PPO (higher data quality).

Training is secondary here; the primary goal is to evaluate RL+cuOpt vs. pure cuOpt.

### 1. Train with PPO (Stable-Baselines3)

In [ ]:
# Config overview: config_pdptw_ppo_train.yaml
with open('config_pdptw_ppo_train.yaml') as f:
    print(f.read())


In [ ]:
%%time
# Train PPO model on n=100 Li&Lim instances
train(config_path='config_pdptw_ppo_train.yaml')


### 2. Train with Tree-Based Pipeline

In [ ]:
# Config overview: config_pdptw_train.yaml
with open('config_pdptw_train.yaml') as f:
    print(f.read())


In [ ]:
%%time
# Train tree_based model on n=100 Li&Lim instances
train(config_path='config_pdptw_train.yaml')


## RL Inference on n=1000 Test Instances

The n=1000 Li&Lim instances are used **exclusively** for testing. The RL model's output here serves as the **initial solution injected into cuOpt**.

### PPO Model Inference

In [ ]:
import yaml

# Point the test config to the PPO model
with open('config_pdptw_test_1000.yaml') as f:
    test_cfg = yaml.safe_load(f)

ppo_model_path = 'earli/pretrained_models/model_pdptw_ppo.m'
test_cfg['train']['pretrained_fname'] = ppo_model_path
test_cfg['logger']['logging_level'] = 20  # verbose

tmp_cfg_ppo = 'config_pdptw_test_1000_ppo_tmp.yaml'
with open(tmp_cfg_ppo, 'w') as f:
    yaml.dump(test_cfg, f)

print('Running PPO inference on n=1000 instances ...')


In [ ]:
%%time
earli_main.main(config_path=tmp_cfg_ppo)

import shutil
shutil.copy('outputs/test_logs.pkl', 'outputs/test_logs_pdptw_1000_ppo.pkl')
print('PPO test log saved to outputs/test_logs_pdptw_1000_ppo.pkl')


### Tree-Based Model Inference

In [ ]:
tree_model_path = 'earli/pretrained_models/model_pdptw.m'

with open('config_pdptw_test_1000.yaml') as f:
    test_cfg_tree = yaml.safe_load(f)

test_cfg_tree['train']['pretrained_fname'] = tree_model_path
test_cfg_tree['logger']['logging_level'] = 20

tmp_cfg_tree = 'config_pdptw_test_1000_tree_tmp.yaml'
with open(tmp_cfg_tree, 'w') as f:
    yaml.dump(test_cfg_tree, f)

print('Running Tree-Based inference on n=1000 instances ...')


In [ ]:
%%time
earli_main.main(config_path=tmp_cfg_tree)

import shutil
shutil.copy('outputs/test_logs.pkl', 'outputs/test_logs_pdptw_1000_tree.pkl')
print('Tree-based test log saved to outputs/test_logs_pdptw_1000_tree.pkl')


## RL-Only Results (Secondary: Training Method Comparison)

These metrics show what the RL models produce **before** any cuOpt refinement.
They also serve as the **initial solution quality** metric in the cuOpt comparison below.

In [ ]:
with open('outputs/test_logs_pdptw_1000_ppo.pkl', 'rb') as f:
    ppo_log = pkl.load(f)

with open('outputs/test_logs_pdptw_1000_tree.pkl', 'rb') as f:
    tree_log = pkl.load(f)

with open('datasets/pdptw_1000.pkl', 'rb') as f:
    problems = pkl.load(f)

def compute_df(log, label, problems):
    best_sols = log['optimal_route']
    n = len(best_sols)
    dm = problems['distance_matrix']
    vehicles = [int(np.sum([node == 0 for node in sol])) - 1 for sol in best_sols]
    costs = [float(utils.solution_cost(sol, dm[i])) for i, sol in enumerate(best_sols)]
    return pd.DataFrame(dict(method=label, problem_id=np.arange(n),
                             vehicles=vehicles, cost=costs))

df_ppo  = compute_df(ppo_log,  'PPO',        problems)
df_tree = compute_df(tree_log, 'Tree-Based', problems)
df_rl = pd.concat([df_ppo, df_tree], ignore_index=True)

print('=== RL-only solution quality (n=1000) ===')
print('PPO:        vehicles={:.2f}  cost={:.4f}'.format(
    df_ppo.vehicles.mean(), df_ppo.cost.mean()))
print('Tree-Based: vehicles={:.2f}  cost={:.4f}'.format(
    df_tree.vehicles.mean(), df_tree.cost.mean()))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
sns.barplot(data=df_rl, x='method', y='vehicles', ax=axes[0])
axes[0].set_title('Mean # Vehicles — RL Only (PDPTW n=1000)')
sns.barplot(data=df_rl, x='method', y='cost', ax=axes[1])
axes[1].set_title('Mean Cost — RL Only (PDPTW n=1000)')
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/pdptw_1000_rl_only_comparison.png', dpi=120)
plt.show()


## Primary Comparison: Pure cuOpt vs. RL-Initialized cuOpt

Three runs:
1. **cuOpt** — pure cuOpt with no RL initialization (baseline).
2. **PPO+cuOpt** — cuOpt seeded with the PPO RL model's solutions.
3. **Tree+cuOpt** — cuOpt seeded with the Tree-based RL model's solutions.

**Note**: the method name for RL-injected cuOpt runs must contain the string `'RL'` so that the injection engine loads the solution file. We use `'RL+cuOpt'` internally and rename to `'PPO+cuOpt'` / `'Tree+cuOpt'` for display.

### Run 1: Pure cuOpt (Baseline — No RL Initialization)

In [ ]:
import yaml

with open('config_pdptw_cuopt_1000.yaml') as f:
    cuopt_cfg_pure = yaml.safe_load(f)

# 'cuOpt' does not contain 'RL', so no solutions are loaded → pure cuOpt.
cuopt_cfg_pure['names']['methods']      = ['cuOpt']
cuopt_cfg_pure['names']['main_method']  = 'cuOpt'
cuopt_cfg_pure['names']['baseline']     = 'cuOpt'
cuopt_cfg_pure['names']['run_title']    = 'cuopt_pure'
cuopt_cfg_pure['paths']['solutions']    = None
cuopt_cfg_pure['paths']['out_summary']  = 'pdptw_1000'

tmp_cuopt_pure = 'config_pdptw_cuopt_pure_tmp.yaml'
with open(tmp_cuopt_pure, 'w') as f:
    yaml.dump(cuopt_cfg_pure, f)

print('Running pure cuOpt on n=1000 instances (no RL initialization) ...')


In [ ]:
%%time
cuopt_main(config_path=tmp_cuopt_pure)
print('Pure cuOpt summary saved to outputs/pdptw_1000_cuopt_pure.pkl')


### Run 2: cuOpt + PPO Initialization

In [ ]:
with open('config_pdptw_cuopt_1000.yaml') as f:
    cuopt_cfg_ppo = yaml.safe_load(f)

# IMPORTANT: method name MUST contain 'RL' for solutions to be loaded into cuOpt.
cuopt_cfg_ppo['names']['methods']      = ['RL+cuOpt']
cuopt_cfg_ppo['names']['main_method']  = 'RL+cuOpt'
cuopt_cfg_ppo['names']['baseline']     = 'RL+cuOpt'
cuopt_cfg_ppo['names']['run_title']    = 'cuopt_ppo'
cuopt_cfg_ppo['paths']['solutions']    = 'outputs/test_logs_pdptw_1000_ppo.pkl'
cuopt_cfg_ppo['paths']['out_summary']  = 'pdptw_1000'

tmp_cuopt_ppo = 'config_pdptw_cuopt_ppo_tmp.yaml'
with open(tmp_cuopt_ppo, 'w') as f:
    yaml.dump(cuopt_cfg_ppo, f)

print('Running cuOpt + PPO initialization on n=1000 instances ...')


In [ ]:
%%time
cuopt_main(config_path=tmp_cuopt_ppo)
print('PPO+cuOpt summary saved to outputs/pdptw_1000_cuopt_ppo.pkl')


### Run 3: cuOpt + Tree-Based Initialization

In [ ]:
with open('config_pdptw_cuopt_1000.yaml') as f:
    cuopt_cfg_tree = yaml.safe_load(f)

cuopt_cfg_tree['names']['methods']      = ['RL+cuOpt']
cuopt_cfg_tree['names']['main_method']  = 'RL+cuOpt'
cuopt_cfg_tree['names']['baseline']     = 'RL+cuOpt'
cuopt_cfg_tree['names']['run_title']    = 'cuopt_tree'
cuopt_cfg_tree['paths']['solutions']    = 'outputs/test_logs_pdptw_1000_tree.pkl'
cuopt_cfg_tree['paths']['out_summary']  = 'pdptw_1000'

tmp_cuopt_tree = 'config_pdptw_cuopt_tree_tmp.yaml'
with open(tmp_cuopt_tree, 'w') as f:
    yaml.dump(cuopt_cfg_tree, f)

print('Running cuOpt + Tree-Based initialization on n=1000 instances ...')


In [ ]:
%%time
cuopt_main(config_path=tmp_cuopt_tree)
print('Tree+cuOpt summary saved to outputs/pdptw_1000_cuopt_tree.pkl')


## Per-Problem Results

The table below shows, **for every test instance**, the solution quality of:
* `ppo_init` / `tree_init` — RL initial solution **before cuOpt refinement**
* `cuOpt` — pure cuOpt (no RL)
* `PPO+cuOpt` / `Tree+cuOpt` — cuOpt **after** RL initialization

This mirrors the per-instance output of `test_injection.py`.

In [ ]:
# Load all three cuOpt run summaries
with open('outputs/pdptw_1000_cuopt_pure.pkl', 'rb') as f:
    pure_data = pkl.load(f)
with open('outputs/pdptw_1000_cuopt_ppo.pkl', 'rb') as f:
    ppo_data = pkl.load(f)
with open('outputs/pdptw_1000_cuopt_tree.pkl', 'rb') as f:
    tree_data = pkl.load(f)

df_cuopt   = pure_data['summary']               # method == 'cuOpt'
df_ppo_co  = ppo_data['summary'].copy()         # method == 'RL+cuOpt'
df_tree_co = tree_data['summary'].copy()        # method == 'RL+cuOpt'

# Rename for clarity in merged tables
df_ppo_co['method']  = 'PPO+cuOpt'
df_tree_co['method'] = 'Tree+cuOpt'

# RL initial solution quality stored in info['external_costs'].
# Keys: {'vehicles': {'RL+cuOpt': [v_0, v_1, ...]},
#        'costs':    {'RL+cuOpt': [c_0, c_1, ...]}}
# Indexed by problem_id (dataset index).
ppo_init_v  = ppo_data['info']['external_costs']['vehicles']['RL+cuOpt']
ppo_init_c  = ppo_data['info']['external_costs']['costs']['RL+cuOpt']
tree_init_v = tree_data['info']['external_costs']['vehicles']['RL+cuOpt']
tree_init_c = tree_data['info']['external_costs']['costs']['RL+cuOpt']

print('Loaded summaries. Problems in pure cuOpt run:',
      df_cuopt['problem_id'].nunique())


In [ ]:
# Build per-problem flat table -----------------------------------------------
def _extract(df, vcol, ccol):
    return (df[['problem_id', 'vehicles', 'cost']]
              .rename(columns={'vehicles': vcol, 'cost': ccol}))

per_problem = (
    _extract(df_cuopt,   'cuopt_vehicles',      'cuopt_cost')
    .merge(_extract(df_ppo_co,  'ppo_cuopt_vehicles', 'ppo_cuopt_cost'),
           on='problem_id')
    .merge(_extract(df_tree_co, 'tree_cuopt_vehicles','tree_cuopt_cost'),
           on='problem_id')
)

# Attach RL initial quality (before cuOpt runs)
per_problem['ppo_init_vehicles']  = [
    ppo_init_v[pid]  for pid in per_problem['problem_id']]
per_problem['ppo_init_cost']      = [
    float(ppo_init_c[pid])  for pid in per_problem['problem_id']]
per_problem['tree_init_vehicles'] = [
    tree_init_v[pid] for pid in per_problem['problem_id']]
per_problem['tree_init_cost']     = [
    float(tree_init_c[pid]) for pid in per_problem['problem_id']]

# Reorder columns for readability
col_order = [
    'problem_id',
    'ppo_init_vehicles',  'ppo_init_cost',
    'tree_init_vehicles', 'tree_init_cost',
    'cuopt_vehicles',     'cuopt_cost',
    'ppo_cuopt_vehicles', 'ppo_cuopt_cost',
    'tree_cuopt_vehicles','tree_cuopt_cost',
]
per_problem = per_problem[col_order].sort_values('problem_id').reset_index(drop=True)

print(f'Per-problem table: {len(per_problem)} instances')
per_problem


In [ ]:
# Full per-problem table as printed output (mirrors test_injection style)
print('=== Per-Instance Results (PDPTW n=1000) ===')
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', 0, 'display.float_format', '{:.4f}'.format):
    print(per_problem.to_string(index=False))


## Summary Statistics

In [ ]:
# Combine all methods into one DataFrame for aggregation
df_combined = pd.concat(
    [df_ppo.rename(columns={'vehicles':'vehicles','cost':'cost'})[['method','problem_id','vehicles','cost']],
     df_tree.rename(columns={'vehicles':'vehicles','cost':'cost'})[['method','problem_id','vehicles','cost']],
     df_cuopt[['method','problem_id','vehicles','cost']],
     df_ppo_co[['method','problem_id','vehicles','cost']],
     df_tree_co[['method','problem_id','vehicles','cost']]],
    ignore_index=True)

summary = (df_combined.groupby('method')
           .agg(mean_vehicles=('vehicles','mean'), mean_cost=('cost','mean'))
           .reset_index())
print('=== Solution Quality Summary (PDPTW n=1000) ===')
print(summary.to_string(index=False))


In [ ]:
# Primary bar chart: cuOpt alone vs. RL-initialized cuOpt
order_primary = ['cuOpt', 'PPO+cuOpt', 'Tree+cuOpt']
df_primary = df_combined[df_combined.method.isin(order_primary)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=df_primary, x='method', y='vehicles',
            order=order_primary, ax=axes[0])
axes[0].set_title('Mean # Vehicles (PDPTW n=1000)\n(Primary: cuOpt vs. RL+cuOpt)')
axes[0].tick_params(axis='x', rotation=10)

sns.barplot(data=df_primary, x='method', y='cost',
            order=order_primary, ax=axes[1])
axes[1].set_title('Mean Cost (PDPTW n=1000)\n(Primary: cuOpt vs. RL+cuOpt)')
axes[1].tick_params(axis='x', rotation=10)

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/pdptw_1000_primary_comparison.png', dpi=120)
plt.show()


In [ ]:
# Full comparison: all 5 methods (RL-only → final cuOpt)
order_full = ['PPO', 'Tree-Based', 'cuOpt', 'PPO+cuOpt', 'Tree+cuOpt']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=df_combined, x='method', y='vehicles',
            order=order_full, ax=axes[0])
axes[0].set_title('Mean # Vehicles (PDPTW n=1000) — All Methods')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=df_combined, x='method', y='cost',
            order=order_full, ax=axes[1])
axes[1].set_title('Mean Cost (PDPTW n=1000) — All Methods')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('outputs/pdptw_1000_full_comparison.png', dpi=120)
plt.show()


In [ ]:
# Improvement statistics: how much does RL initialization help cuOpt?
cuopt_mean_cost  = df_cuopt['cost'].mean()
ppo_co_mean_cost = df_ppo_co['cost'].mean()
tree_co_mean_cost= df_tree_co['cost'].mean()

print('=== RL Initialization Improvement over Pure cuOpt ===')
print(f'Pure cuOpt mean cost:          {cuopt_mean_cost:.4f}')
print(f'PPO+cuOpt  mean cost:          {ppo_co_mean_cost:.4f}  '
      f'(improvement: {100*(cuopt_mean_cost-ppo_co_mean_cost)/cuopt_mean_cost:.2f}%)')
print(f'Tree+cuOpt mean cost:          {tree_co_mean_cost:.4f}  '
      f'(improvement: {100*(cuopt_mean_cost-tree_co_mean_cost)/cuopt_mean_cost:.2f}%)')

print()
print('=== RL Initial → cuOpt Final Improvement (PPO) ===')
ppo_init_mean = per_problem['ppo_init_cost'].mean()
print(f'PPO initial (before cuOpt):    {ppo_init_mean:.4f}')
print(f'PPO+cuOpt  final:              {ppo_co_mean_cost:.4f}  '
      f'(cuOpt gain: {100*(ppo_init_mean-ppo_co_mean_cost)/ppo_init_mean:.2f}%)')

print()
print('=== RL Initial → cuOpt Final Improvement (Tree) ===')
tree_init_mean = per_problem['tree_init_cost'].mean()
print(f'Tree initial (before cuOpt):   {tree_init_mean:.4f}')
print(f'Tree+cuOpt  final:             {tree_co_mean_cost:.4f}  '
      f'(cuOpt gain: {100*(tree_init_mean-tree_co_mean_cost)/tree_init_mean:.2f}%)')


In [ ]:
# Scatter plot: RL initial cost vs cuOpt final cost (per problem)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (init_col, final_col, label) in zip(
        axes,
        [('ppo_init_cost',  'ppo_cuopt_cost',  'PPO'),
         ('tree_init_cost', 'tree_cuopt_cost', 'Tree-Based')]):
    ax.scatter(per_problem[init_col], per_problem[final_col],
               alpha=0.6, s=20, label=f'{label}')
    lims = [min(per_problem[init_col].min(), per_problem[final_col].min()),
            max(per_problem[init_col].max(), per_problem[final_col].max())]
    ax.plot(lims, lims, 'r--', lw=1, label='no improvement')
    ax.set_xlabel(f'RL initial cost ({label})')
    ax.set_ylabel(f'{label}+cuOpt final cost')
    ax.set_title(f'Initial vs Final Cost ({label})\n'
                  f'Points below diagonal = cuOpt improved the RL solution')
    ax.legend()

plt.tight_layout()
plt.savefig('outputs/pdptw_1000_init_vs_final_cost.png', dpi=120)
plt.show()


## Cleanup

In [ ]:
for f in [tmp_cfg_ppo, tmp_cfg_tree,
          tmp_cuopt_pure, tmp_cuopt_ppo, tmp_cuopt_tree]:
    if os.path.exists(f):
        os.remove(f)
        print(f'Removed {f}')
